In [ ]:
"""Comprehensive zero-shot evaluation using the centralized benchmark registry.
This mirrors plot_confusion_matrix_and_get_performance_metrics from fig2 but uses
the registry-based approach for DRY compliance."""

In [ ]:
import warnings
# avoid DeprecationWarning: np.find_common_type is deprecated due to pandas version
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pandas.core.algorithms")

In [ ]:
import pandas as pd
import logging
import numpy as np
import copy
import matplotlib
import torch
import sys
from pathlib import Path

# Add src to path for imports
sys.path.append(str(Path.cwd().parent.parent / "src"))

from cellwhisperer.validation.zero_shot.functions import (
    get_performance_metrics_transcriptome_vs_text
)
from cellwhisperer.validation.registry import ValidationRegistry
from cellwhisperer.utils.model_io import load_cellwhisperer_model

# Import utility functions from figures pipeline
sys.path.append(str(Path.cwd().parent.parent / "src" / "figures" / "notebooks"))
from zero_shot_validation_scripts.utils import SUFFIX_PREFIX_DICT
from zero_shot_validation_scripts.dataset_preparation import load_and_preprocess_dataset

logging.basicConfig(level=logging.INFO)

In [ ]:
#### Parameters ####

matplotlib.style.use(snakemake.input.mpl_style)

dataset_name = snakemake.wildcards.dataset
metadata_col = snakemake.wildcards.metadata_col
model_name = snakemake.wildcards.model

# Find the matching benchmark specification from the registry
benchmarks = ValidationRegistry.get_cellwhisperer_benchmarks()
benchmark_spec = None

for spec in benchmarks:
    if spec.dataset == dataset_name and spec.metadata_col == metadata_col:
        benchmark_spec = spec
        break

if benchmark_spec is None:
    # Fallback if not found in registry
    logging.warning(f"Benchmark for {dataset_name}/{metadata_col} not found in registry, using defaults")
    benchmark_spec = type('BenchmarkSpec', (), {
        'dataset': dataset_name,
        'metadata_col': metadata_col,
        'dataset_kwargs': {'celltype_obs_colname': metadata_col},
        'processor_kwargs': {},
        'description': f'{dataset_name} {metadata_col} prediction'
    })()

logging.info(f"Running comprehensive evaluation for: {benchmark_spec.description}")
logging.info(f"Dataset: {dataset_name}, Metadata column: {metadata_col}")

In [ ]:
# Load model
(
    pl_model_cellwhisperer,
    text_processor_cellwhisperer,
    cellwhisperer_transcriptome_processor,
) = load_cellwhisperer_model(model_path=snakemake.input.model, eval=True)
cellwhisperer_model = pl_model_cellwhisperer.model

logging.info(f"Model loaded: {model_name}")

In [ ]:
#### Load data
adata = load_and_preprocess_dataset(
    dataset_name=dataset_name, 
    read_count_table_path=snakemake.input.raw_read_count_table,
    obsm_paths={
        "X_cellwhisperer": (snakemake.input.processed_dataset, "transcriptome_embeds"),
        "X_geneformer": (snakemake.input.processed_dataset, "transcriptome_features")
    }
)
logging.info(f"Data loaded and preprocessed. Shape: {adata.shape}")

In [ ]:
# Filter out NaN values in the metadata column
adata_no_nans = adata[
    ~(adata.obs[metadata_col].isna()) & ~(adata.obs[metadata_col] == "nan")
].copy()

logging.info(f"After removing NaNs: {adata_no_nans.shape}")

# Prepare text list with proper prefix/suffix if available
if snakemake.params.use_prefix_suffix_version and metadata_col in SUFFIX_PREFIX_DICT:
    prefix, suffix = SUFFIX_PREFIX_DICT[metadata_col]
    text_list = [f"{prefix}{x}{suffix}" for x in adata_no_nans.obs[metadata_col].unique().tolist()]
    logging.info(f"Using prefix/suffix for {metadata_col}: '{prefix}' + label + '{suffix}'")
elif metadata_col not in SUFFIX_PREFIX_DICT:
    logging.warning(f"Label column {metadata_col} not found in SUFFIX_PREFIX_DICT, continuing without prefix/suffix")
    text_list = adata_no_nans.obs[metadata_col].unique().tolist()
else:
    text_list = adata_no_nans.obs[metadata_col].unique().tolist()

logging.info(f"Created text list with {len(text_list)} unique labels")

In [ ]:
#### Get classification performance metrics for cellwhisperer
    
correct_text_idx_per_transcriptome = [
    adata_no_nans.obs[metadata_col].unique().tolist().index(x)
    for x in adata_no_nans.obs[metadata_col].values
]

logging.info("Computing comprehensive performance metrics...")

(
    performance_metrics,
    performance_metrics_per_label_df,
) = get_performance_metrics_transcriptome_vs_text(
    model=cellwhisperer_model,
    modality_input=torch.tensor(adata_no_nans.obsm["X_cellwhisperer"], device=cellwhisperer_model.device),
    text_list_or_text_embeds=text_list,
    average_mode=None,
    grouping_keys=None,
    transcriptome_processor=cellwhisperer_transcriptome_processor,
    batch_size=32,
    score_norm_method=None,
    correct_text_idx_per_transcriptome=correct_text_idx_per_transcriptome,
)

logging.info(f"Performance metrics computed. Macro F1: {performance_metrics.get('f1', 'N/A'):.3f}, Accuracy: {performance_metrics.get('accuracy', 'N/A'):.3f}")

In [ ]:
# Save results
pd.Series(performance_metrics).to_csv(snakemake.output.performance_metrics)
performance_metrics_per_label_df.to_csv(snakemake.output.performance_metrics_per_metadata)

logging.info(f"Results saved to:")
logging.info(f"  - {snakemake.output.performance_metrics}")
logging.info(f"  - {snakemake.output.performance_metrics_per_metadata}")

# Log summary statistics
logging.info("\n=== COMPREHENSIVE EVALUATION SUMMARY ===")
logging.info(f"Dataset: {dataset_name}")
logging.info(f"Metadata column: {metadata_col}")
logging.info(f"Model: {model_name}")
logging.info(f"Number of cells: {adata_no_nans.shape[0]}")
logging.info(f"Number of classes: {len(text_list)}")
logging.info(f"Classes: {adata_no_nans.obs[metadata_col].unique().tolist()}")
logging.info(f"Performance metrics:")
for metric, value in performance_metrics.items():
    if isinstance(value, (int, float)):
        logging.info(f"  {metric}: {value:.4f}")
    else:
        logging.info(f"  {metric}: {value}")
logging.info("=========================================\n")